# TaskB: Matching the Job-Title with the Relevant Skills using ESCO Ontology

In this notebook, we prepare the shared-track Task-B dataset 
- for expanding the job-title and skills using Large langauge models (LLMs) through prompting
- for matching the job-title to Skills via original and expanded forms
- for evaluating the predicted matched skills using evaluation script [task's evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script).

- Additionally, the provided format is also compatible with the testset.

## Import Required Libraries

In [1]:
import os
import sys
import json

In [2]:
import pandas as pd
import numpy as np
import subprocess
import ast

from tqdm import tqdm

In [3]:
from sentence_transformers import SentenceTransformer, util
from codecarbon import EmissionsTracker

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

## Matching the JobTitle for Validation data 
- Preparing data (Task B)

### Setting up data path

In [5]:
root_data_path="taskB"

In [6]:
#Training data path
taskB_training_path = os.path.join(root_data_path, "training")

#validation data path
validation_data_path=os.path.join(root_data_path, "validation")

#testing data path
test_data_path=os.path.join(root_data_path, "test")

#model path
models_path = "models"

#results path
results_path = "results"

#evaluation result path
evaluations_path = "evaluations"

### Training data loading

#### Training data dictionary

The training data for Task B is shared in 3 files:
- `job2skill.tsv`: This file has been curated to include the most representative skills for each job title in <a href="https://esco.ec.europa.eu/en">ESCO</a>. This file contains three columns:
  - `job_id`: ESCO identifier for the job position.
  - `skill_id`: ESCO identifier for the skill.
  - `rel_type`: Indicator specifying whether the skill_id is essential or optional for a specific job_id. It can have the value “essential” or “optional.”

- `jobid2terms.json`: This JSON file contains job_id identifiers used in the training set for Task B as keys, and a list of valid lexical variants for each identifier as values.

- `skillid2terms.json`: This JSON file contains skill_id identifiers as keys, and a list of valid lexical variants for each identifier as values.


#### Reading Job2Skill data file

In [ ]:
# Read job2skill file
job2skill = pd.read_csv(os.path.join(taskB_training_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])

print ("No. of job_id and skill_id triple: {}".format(job2skill.shape[0]))
job2skill.head()

#### Reading Job2Terms file

In [ ]:
# Read json files
jobid2terms_path = os.path.join(taskB_training_path, "jobid2terms.json")
print (jobid2terms_path)

In [ ]:
with open(os.path.join(jobid2terms_path), 'r') as file:
    jobid2terms = json.load(file)

In [ ]:
print ("Total job ids: {}".format(len(jobid2terms)))

In [ ]:
jobid2terms

#### Reading Skilled2Terms

In [ ]:
# Read json files
skillid2terms_path = os.path.join(taskB_training_path, "skillid2terms.json")
print (skillid2terms_path)

In [ ]:
with open(os.path.join(skillid2terms_path), 'r') as file:
    skillid2terms = json.load(file)

In [ ]:
print ("Total skill ids: {}".format(len(skillid2terms)))

#### Play with Job-id, Skill-id, Job-id-terms, and Skill-id-terms. 

##### Expanding Job2Skill with Terms Mapping

In [ ]:
job2skill["job_terms"] = job2skill["job_id"].map(jobid2terms)
job2skill["skill_terms"] = job2skill["skill_id"].map(skillid2terms)

In [ ]:
job2skill.describe()

**Let's see the skills for a specific job_id.**



In [ ]:
job_id_to_search = job2skill.job_id.to_list()[100]
job_data = job2skill[job2skill.job_id == job_id_to_search]

**The job titles of the selected job_id are:**

In [ ]:
job_data.job_terms.iloc[0]

**For that job_id we fin 9 essential skills and 1 optional skill:**

In [ ]:
job_data.rel_type.value_counts()

The list of skills related to the job_id are:

In [ ]:
job_data.skill_terms.to_list()[0]

In [ ]:
job2skill.job_id.to_list()[100]

### **Validation data loading**

#### **Validation data path setup**

In [7]:
validation_queries_path = os.path.join(validation_data_path, "queries")
validation_corpus_elements_path = os.path.join(validation_data_path, "corpus_elements")

#### **Load queries and corpus elements**

In [8]:
validation_queries = pd.read_csv(validation_queries_path, sep="\t")
validation_corpus_elements = pd.read_csv(validation_corpus_elements_path, sep="\t")

In [9]:
print ("Number of queries and copurs elements in validation dataset: {} and {}".format(len(validation_queries), len(validation_corpus_elements)))

Number of queries and copurs elements in validation dataset: 304 and 1439


In [10]:
print ("validation query samples:")
validation_queries.head()

validation query samples:


,q_id,jobtitle
0,dev_qb_jt_1,corporate governance analyst
1,dev_qb_jt_2,logistics business analyst
2,dev_qb_jt_3,operations planning analyst
3,dev_qb_jt_4,real estate data analyst
4,dev_qb_jt_5,security analyst ii


In [11]:
validation_queries.jobtitle[0]

'corporate governance analyst'

In [12]:
print ("validation corpus elements samples:")
validation_corpus_elements.head()

validation corpus elements samples:


,c_id,esco_uri,skill_aliases
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"['pricing plans', 'price strategies', 'pricing..."
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"['putting out fires', 'put out fires', 'exting..."
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"['online assessment', 'analysis of web strateg..."
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"['keeping up with trends', 'keep pace with tre..."
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"['prepare tax returns form', 'complete tax ret..."


In [13]:
validation_corpus_elements.skill_aliases[0]

"['pricing plans', 'price strategies', 'pricing tactics', 'pricing strategies', 'pricing strategy']"

### **Matching JobTitle with Skills**

- T1: Preprocessing: Aggregate the skill aliases into one long text
- T2: Expand the job-title and skills using LLMs as data augmentation
- T3: Match the expanded job-title text with the skills

  - Prompt Qwen model for title and skill expansion
  - Dense representation using SBERT
  - Relevance computation using Cosine similarity

##### **Transform `skill_aliases` column to a list of strings**

In [14]:
validation_corpus_elements["skill_aliases"] = validation_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

In [15]:
validation_corpus_elements.head()

,c_id,esco_uri,skill_aliases
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"[pricing plans, price strategies, pricing tact..."
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"[putting out fires, put out fires, extinguish ..."
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"[online assessment, analysis of web strategy, ..."
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"[keeping up with trends, keep pace with trends..."
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"[prepare tax returns form, complete tax return..."


##### **Aggregate the skill aliases into one long text**

In [16]:
def aggregate_list_of_terms(terms_list):
    '''
        aggregate the list of terms to a long text (string)
    '''
    return " ".join(terms_list)

In [17]:
validation_corpus_elements["skill_document"] = validation_corpus_elements["skill_aliases"].apply(lambda x: aggregate_list_of_terms(x))

In [18]:
validation_corpus_elements.head()

,c_id,esco_uri,skill_aliases,skill_document
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"[pricing plans, price strategies, pricing tact...",pricing plans price strategies pricing tactics...
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"[putting out fires, put out fires, extinguish ...",putting out fires put out fires extinguish fir...
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"[online assessment, analysis of web strategy, ...",online assessment analysis of web strategy web...
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"[keeping up with trends, keep pace with trends...",keeping up with trends keep pace with trends f...
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"[prepare tax returns form, complete tax return...",prepare tax returns form complete tax returns ...


In [19]:
validation_corpus_elements.skill_document[0]

'pricing plans price strategies pricing tactics pricing strategies pricing strategy'

#### Generating a mapping (dictionary) for queries

In [20]:
'''
    build a dictionary/mapping between the query_id and text from the query pandas dataframe
'''
validation_queries_ids = validation_queries.q_id.to_list()
validation_queries_texts = validation_queries.jobtitle.to_list()

validation_queries_map = dict(zip(validation_queries_ids, validation_queries_texts))

In [21]:
list(validation_queries_map.items())[0]

('dev_qb_jt_1', 'corporate governance analyst')

In [22]:
validation_corpus_elements.head()

,c_id,esco_uri,skill_aliases,skill_document
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"[pricing plans, price strategies, pricing tact...",pricing plans price strategies pricing tactics...
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"[putting out fires, put out fires, extinguish ...",putting out fires put out fires extinguish fir...
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"[online assessment, analysis of web strategy, ...",online assessment analysis of web strategy web...
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"[keeping up with trends, keep pace with trends...",keeping up with trends keep pace with trends f...
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"[prepare tax returns form, complete tax return...",prepare tax returns form complete tax returns ...


#### Generaring a mapping for Documents (Corpus elements)

In [23]:
'''
    build a dictionary/mapping between the corpus_id and document from the pandas dataframe
'''
validation_documents_ids = validation_corpus_elements.c_id.to_list()
validation_documents_esco_uri = validation_corpus_elements.esco_uri.to_list()
validation_documents_texts = validation_corpus_elements.skill_document.to_list()

validation_documents_map = dict(zip(validation_documents_ids, validation_documents_texts))

In [24]:
list(validation_documents_map.items())[0]

('dev_cb_sk_1',
 'pricing plans price strategies pricing tactics pricing strategies pricing strategy')

#### Prompting Large Language Models for Data Augmentation

In [ ]:
#query_text = "corporate governance analyst"
#doc_text = "pricing plans price strategies pricing tactics pricing strategies pricing strategy"

In [ ]:
#model_name = "Qwen/Qwen3-8B"
#model_name = "Qwen/Qwen2.5-7B"
model_name = "Qwen/Qwen2.5-7B-Instruct"

# LLM and SBERT are used in zero-shot learning setting. We leave the fine-tuning task as future work.

#### **Loading Large Language Model**

In [ ]:
#model = SentenceTransformer("all-MiniLM-L6-v2", token=False)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
)

tokenizer =  AutoTokenizer.from_pretrained(model_name)

In [ ]:
def get_job_prompt(query_text):
    '''
        create a prompt for augmenting the job title in terms of the skills required.
    '''
    job_prompt = "Given {}, briefly (1-2 sentences) describe the skills needed for that job".format(query_text) 

    job_message = [
        {"role": "user", "content": job_prompt},
    ]

    return job_message

In [ ]:
def get_skill_prompt(query_text):
    '''
        create a prompt for augmenting the skills in terms of the job/occupation role that require that skills.
    '''
    skill_prompt = "Given {}, briefly (1-2 sentences) describe the job roles that require that skill".format(query_text)

    skill_message = [
        {"role": "user", "content": skill_prompt},
    ]

    return skill_message

In [ ]:
def get_tokenized_prompt(prompt, tokenization=False, generation_prompt=True, thinking=False):
    tokenized_prompt = tokenizer.apply_chat_template(
        prompt,
        tokenize=tokenization,
        add_generation_prompt=generation_prompt,
        enable_thinking=thinking, # Switches between thinking and non-thinking modes. Default is True.
    )
    return tokenized_prompt

In [ ]:
def get_model_input(tokenized_prompt, tokenizer, model):
    model_inputs = tokenizer([tokenized_prompt], return_tensors="pt").to(model.device)

    return model_inputs

In [ ]:
def get_generated_ids(model_inputs, tokenizer, model):
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    return generated_ids

In [ ]:
def get_decoded_output(generated_ids, tokenizer):
    content = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return content

In [ ]:
def get_llm_augmented_query(query_text, tokenizer, model):
    job_prompt = get_job_prompt(query_text)
    print (job_prompt)
    
    prompt_tokenized = get_tokenized_prompt(job_prompt, tokenization=False, generation_prompt=True, thinking=False)
    print (prompt_tokenized)

    model_inputs = get_model_input(prompt_tokenized, tokenizer, model)
    print (model_inputs)

    generated_ids = get_generated_ids(model_inputs)
    generated_content = get_decoded_output(generated_ids, tokenizer)

    return generated_content

In [ ]:
def get_llm_augmented_document(document_text, model, tokenizer):
    skill_prompt = get_skill_prompt(document_text)
    skill_prompt_tokenized = get_tokenized_prompt(skill_prompt, tokenization=False, generation_prompt=True, thinking=False)
    model_input = get_model_input(skill_prompt_tokenized, tokenizer, model)
    generated_ids = get_generated_ids(model_inputs)
    generated_content = get_decoded_output(generated_ids, tokenizer)

    return generated_content

In [ ]:
query_augmentation = []
for query_id in validation_queries_ids:
    query_text = validation_queries_map.get(query_id)

    llm_augmented_query = get_llm_augmented_query(query_text, model, tokenizer)
    query_augmentation.append([query_id, query_text, llm_augmented_query])
    
print ("Total augmented query title: {}".format(len(query_augmentation)))

In [ ]:
skill_augmentation = []
for document_id in validation_documents_ids:
    document_text = validation_documents_map.get(document_id)

    llm_augmented_document = get_llm_augmented_document(document_text, model, tokenizer)
    skill_augmentation.append([document_id, document_text, llm_augmented_document])
    
print ("Total augmented documents: {}".format(len(skill_augmentation)))

In [ ]:
### Job-title: "Given a job role (and its synonyms), briefly (1-2 sentences) describe the skills needed for that job". This generated a ‘skill_brief‘ for each job title.
### job-skills: "Given a skill (and its synonyms), briefly (1-2 sentences) describe the job roles thatrequire that skill". This generated a ‘job_brief‘ for each skill

#### Query: job-title augmentation module

In [ ]:
# Query job-title augmentation module

query_augmentation = []
for query_index in range(len(validation_queries_ids)):
    query_id = validation_queries_ids[query_index]
    
    query_text = validation_queries_map.get(query_id)

    job_prompt = get_job_prompt(query_text)
    #print (job_prompt)
    
    job_prompt_tokenized = get_tokenized_prompt(job_prompt, tokenization=False, generation_prompt=True, thinking=False)
    #print (job_prompt_tokenized)

    model_inputs = get_model_input(job_prompt_tokenized, tokenizer, model)
    #print (model_inputs)

    generated_ids = get_generated_ids(model_inputs, tokenizer, model)
    #print (generated_ids)

    generated_content = get_decoded_output(generated_ids, tokenizer)
    #print (generated_content)

    query_augmentation.append([query_id, query_text, generated_content])

print ("Total augmented query title: {}".format(len(query_augmentation)))

In [ ]:
#query_job_title_aug = pd.DataFrame(query_augmentation, columns=['job_id', 'job_title', 'job_augmented'])
#query_job_title_aug.to_csv("taskB/validation/queries_augmented.csv", index=False)

#### Documents: Skills augmentation module

In [ ]:
#doc_text = "pricing plans price strategies pricing tactics pricing strategies pricing strategy"

document_augmentation = []
for doc_index in tqdm(range(len(validation_documents_ids))):
    doc_id = validation_documents_ids[doc_index]
    
    document_text = validation_documents_map.get(doc_id)

    skill_prompt = get_skill_prompt(document_text)
    #print (skill_prompt)
    
    skill_prompt_tokenized = get_tokenized_prompt(skill_prompt, tokenization=False, generation_prompt=True, thinking=False)
    #print (skill_prompt_tokenized)

    model_inputs = get_model_input(skill_prompt_tokenized, tokenizer, model)
    #print (model_inputs)

    generated_ids = get_generated_ids(model_inputs, tokenizer, model)
    #print (generated_ids)

    generated_content = get_decoded_output(generated_ids, tokenizer)
    #print (generated_content)

    document_augmentation.append([doc_id, document_text, generated_content])
    #if doc_index == 0:
    #    break

print ("Total augmented query title: {}".format(len(document_augmentation)))

In [ ]:
document_skills_aug = pd.DataFrame(document_augmentation, columns=['skill_id', 'skill_title', 'skill_augmented'])
document_skills_aug.to_csv("taskB/validation/corpus_elements_augmented.csv", index=False)

In [127]:
#document_skills_aug.head()
print ("{}\n{}\n{}".format(document_skills_aug.skill_id[0], document_skills_aug.skill_title[0], document_skills_aug.skill_augmented[0]))

dev_cb_sk_1
pricing plans price strategies pricing tactics pricing strategies pricing strategy
The job roles that require the skill of understanding and implementing pricing plans, strategies, and tactics include but are not limited to: Pricing Analyst, Pricing Manager, Product Manager, Marketing Manager, Sales Manager, and Financial Analyst. These professionals are responsible for analyzing market trends, competitor pricing, and customer behavior to develop and execute effective pricing strategies that maximize revenue and profitability.


#### **Embedding Queries and Documents**
- Dense representation using SentenceTransformers

In [25]:
sbert = SentenceTransformer("all-MiniLM-L6-v2", token=False)

##### Query-document matching 
Using LLM, we generated short description of the job title (job_brief) and skills (skill_brief). Now we can match the query and document in different combination
    
- A: Job_title (query) + Skill (document)
- B: Job_title + Skill_brief
- C: Job_brief + Skill
- D: Job_brief + Skill_brief

Now, we will represent the query and document using SBERT and compute the cosine similarity.

```sbert = SentenceTransformer("all-MiniLM-L6-v2", token=False)'''

In [27]:
query_job_title_aug = pd.read_csv("taskB/validation/queries_augmented.csv")
document_skills_aug = pd.read_csv("taskB/validation/corpus_elements_augmented.csv")

In [28]:
job_title_text = validation_queries_texts
skill_text = validation_documents_texts

job_brief_text = list(query_job_title_aug.job_augmented)
skill_brief_text = list(document_skills_aug.skill_augmented)

In [37]:
validation_queries_brief_map = dict(zip(validation_queries_ids, job_brief_text))
validation_documents_brief_map = dict(zip(validation_documents_ids, skill_brief_text))

In [29]:
print (len(job_title_text), len(job_brief_text))
print (len(skill_text), len(skill_brief_text))

304 304
1439 1439


In [30]:
job_title_embeddings = sbert.encode(job_title_text, convert_to_tensor=True)
skill_embeddings = sbert.encode(skill_text, convert_to_tensor=True)
job_brief_embeddings = sbert.encode(job_brief_text, convert_to_tensor=True)
skill_brief_embeddings = sbert.encode(skill_brief_text, convert_to_tensor=True)

In [31]:
#validation_query_embeddings = sbert.encode(validation_queries_texts, convert_to_tensor=True)

In [32]:
#validation_query_embeddings.shape

In [33]:
#validation_documents_embeddings = sbert.encode(validation_documents_texts, convert_to_tensor=True)

In [57]:
#query_job_title_aug = pd.read_csv("taskB/validation/queries_augmented.csv")
#document_skills_aug = pd.read_csv("taskB/validation/corpus_elements_augmented.csv")

In [34]:
#validation_documents_embeddings.shape 

job_title_embeddings.shape, skill_embeddings.shape, job_brief_embeddings.shape, skill_brief_embeddings.shape

(torch.Size([304, 384]),
 torch.Size([1439, 384]),
 torch.Size([304, 384]),
 torch.Size([1439, 384]))

##### Cosine Similarity between Embedding Representations

In [52]:
validation_sim_job_skill = util.cos_sim(job_title_embeddings, skill_embeddings).cpu().numpy()
validation_sim_job_skill_brief = util.cos_sim(job_title_embeddings, skill_brief_embeddings).cpu().numpy()
validation_sim_job_brief_skill = util.cos_sim(job_brief_embeddings, skill_embeddings).cpu().numpy()
validation_sim_job_brief_skill_brief = util.cos_sim(job_brief_embeddings, skill_brief_embeddings).cpu().numpy()
validation_sim_job_skill_mean = np.mean([validation_sim_job_skill, validation_sim_job_skill_brief, validation_sim_job_brief_skill, validation_sim_job_brief_skill_brief], axis=0)

In [53]:
validation_sim_job_skill.shape, validation_sim_job_skill_brief.shape, validation_sim_job_brief_skill.shape, validation_sim_job_brief_skill_brief.shape, validation_sim_job_skill_mean.shape

((304, 1439), (304, 1439), (304, 1439), (304, 1439), (304, 1439))

In [54]:
validation_sim_job_skill_mean.shape

(304, 1439)

## Evaluation Preparation
- To compute the evaluation metrics, we will generate a submission file using the query-documents similarity metric.
- The submission file must follow the ``TREC Run File format'', including headers in the output file.
- This means that the file should have 6 space-spearated columns per line, with following information:
    - q_id: query ID.
    - Q0: a constant identifier, usually "Q0".
    - doc_id: id of the retrieved corpus elements (skill_id).
    - rank: position of the document in the ranking.
    - score: relevance score assigned by the model.
    - tag: experiment name

### Preparing Submission File

In [38]:
def get_ranked_result_list(queries_ids, queries_map, documents_ids, documents_map, similarities_query_documents, documents_texts, model_name):
    results = []
    results_name = []
    
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities_query_documents[q_idx])
        used_doc_ids = set()
        rank_counter = 0
        for c_idx in sorted_indices:  # Consider the full list.
            doc_id = documents_ids[c_idx]
            # If doc_id was already processed, go to the next one.
            if doc_id in used_doc_ids:
                continue
            used_doc_ids.add(doc_id)
            rank_counter += 1
    
            query_name = queries_map[q_id]
            doc_name = documents_texts[c_idx]
            score = similarities_query_documents[q_idx, c_idx]
    
            results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} {model_name}")
            results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} {model_name}")

    return results, results_name

In [55]:
results_list_validation_query_doc, results_name_validation_query_doc = get_ranked_result_list(
                                                    validation_queries_ids, validation_queries_map,
                                                    validation_documents_ids, validation_documents_map,
                                                    validation_sim_job_skill, skill_text,
                                                   "sbert_job_skill")

result_jobtitle_skill_taskB = os.path.join(results_path, "validation_sbert_job_skill_taskB.trec")
with open(result_jobtitle_skill_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_list_validation_query_doc))


results_list_validation_query_doc_brief, results_name_validation_query_doc_brief = get_ranked_result_list(
                                                    validation_queries_ids, validation_queries_map,
                                                    validation_documents_ids, validation_documents_brief_map,
                                                    validation_sim_job_skill_brief, skill_brief_text,
                                                   "sbert_job_skill_brief")

result_job_skill_brief_taskB = os.path.join(results_path, "validation_sbert_job_skill_brief_taskB.trec")
with open(result_job_skill_brief_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_list_validation_query_doc_brief))

results_list_validation_query_brief_doc, results_name_validation_query_brief_doc = get_ranked_result_list(
                                                    validation_queries_ids, validation_queries_brief_map,
                                                    validation_documents_ids, validation_documents_map,
                                                    validation_sim_job_brief_skill, skill_text,
                                                   "sbert_job_brief_skill")
result_job_brief_skill_taskB = os.path.join(results_path, "validation_sbert_job_brief_skill_taskB.trec")
with open(result_job_brief_skill_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_list_validation_query_brief_doc))


results_list_validation_query_brief_doc_brief, results_name_validation_query_brief_doc_brief = get_ranked_result_list(
                                                    validation_queries_ids, validation_queries_brief_map,
                                                    validation_documents_ids, validation_documents_brief_map,
                                                    validation_sim_job_brief_skill_brief, skill_brief_text,
                                                   "sbert_job_brief_skill_brief")
result_job_brief_skill_brief_taskB = os.path.join(results_path, "validation_sbert_job_brief_skill_brief_taskB.trec")
with open(result_job_brief_skill_brief_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_list_validation_query_brief_doc_brief))

results_list_validation_query_doc_mean, results_name_validation_query_doc_mean = get_ranked_result_list(
                                                    validation_queries_ids, validation_queries_brief_map,
                                                    validation_documents_ids, validation_documents_brief_map,
                                                    validation_sim_job_skill_mean, skill_brief_text,
                                                   "sbert_job_skill_mean")
result_job_skill_mean_taskB = os.path.join(results_path, "validation_sbert_job_skill_mean_taskB.trec")
with open(result_job_skill_mean_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_list_validation_query_doc_mean))

### Evaluation Scripts

- For the evaluation, we will use the official [TalentCLEF evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script), which uses the Ranx library under the hood.

- First, clone the repo and install the requirements file:

In [ ]:
# run this cell just once for the first time
!git clone https://github.com/TalentCLEF/talentclef25_evaluation_script.git
!pip install -r talentclef25_evaluation_script/requirements.txt

Then, select the Qrels file and the Run file to perform the evaluation.

In [56]:
qrels_file = os.path.join(validation_data_path, "qrels.tsv")

run_validation_job_skill_file = os.path.join(results_path, "validation_sbert_job_skill_taskB.trec")

run_validation_job_skill_brief_file = os.path.join(results_path, "validation_sbert_job_skill_brief_taskB.trec")

run_validation_job_brief_skill_file = os.path.join(results_path, "validation_sbert_job_brief_skill_taskB.trec")

run_validation_job_brief_skill_brief_file = os.path.join(results_path, "validation_sbert_job_brief_skill_brief_taskB.trec")

run_validation_job_skill_mean_file = os.path.join(results_path, "validation_sbert_job_skill_mean_taskB.trec")

In [57]:
print ("Evaluation")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_validation_job_skill_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_validation_job_skill_brief_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_validation_job_brief_skill_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_validation_job_brief_skill_brief_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_validation_job_skill_mean_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

Evaluation


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Received parameters:
  qrels: taskB/validation/qrels.tsv
  run: results/validation_sbert_job_skill_taskB.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.1755
mrr: 0.6756
ndcg: 0.6564
precision@5: 0.4336
precision@10: 0.3766
precision@100: 0.1859



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Received parameters:
  qrels: taskB/validation/qrels.tsv
  run: results/validation_sbert_job_skill_brief_taskB.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.2187
mrr: 0.7528
ndcg: 0.6906
precision@5: 0.4954
precision@10: 0.4431
precision@100: 0.2282



...

Received parameters:
  qrels: taskB/validation/qrels.tsv
  run: results/validation_sbert_job_brief_skill_taskB.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.1830
mrr: 0.7054
ndcg: 0.6649
precision@5: 0.4467
precision@10: 0.3757
precision@100: 0.1943



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Received parameters:
  qrels: taskB/validation/qrels.tsv
  run: results/validation_sbert_job_brief_skill_brief_taskB.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.2217
mrr: 0.7158
ndcg: 0.6905
precision@5: 0.4684
precision@10: 0.4207
precision@100: 0.2327



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Received parameters:
  qrels: taskB/validation/qrels.tsv
  run: results/validation_sbert_job_skill_mean_taskB.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.2323
mrr: 0.7425
ndcg: 0.7016
precision@5: 0.5184
precision@10: 0.4691
precision@100: 0.2386



In [ ]:
# Alternative approach for computing evaluation metrics
!python talentclef25_evaluation_script/talentclef_evaluate.py --qrels taskB/validation/qrels.tsv --run results/validation_adhoc_STC_taskB.trec